In [ ]:
import os
import random
import numpy as np
import pandas as pd
import pywt
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import butter, filtfilt, resample
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def butter_bandpass_filter(data, lowcut=50.0, highcut=3000.0, fs=20000.0, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band', analog=False)
    y = filtfilt(b, a, data)
    return y


In [ ]:

class TargetDataset(Dataset):
    def __init__(self, data_list, labels, original_fs, target_fs=10000.0, segment_length=400, transform=None, amp_bounds=None):
        self.samples = []
        self.transform = transform
        self.target_fs = target_fs
        
        for signal, label in zip(data_list, labels):
            filtered_signal = butter_bandpass_filter(signal, lowcut=50.0, highcut=3000.0, fs=original_fs)
            
            if target_fs < original_fs:
                num_samples = int(len(filtered_signal) * (target_fs / original_fs))
                resampled_signal = resample(filtered_signal, num_samples)
            else:
                resampled_signal = filtered_signal
                
            num_segments = len(resampled_signal) // segment_length
            for i in range(num_segments):
                start_idx = i * segment_length
                segment = resampled_signal[start_idx : start_idx + segment_length]
                self.samples.append((segment, label))

        if amp_bounds is not None:
            self.amp_bounds = amp_bounds
        else:
            self.amp_bounds = self._compute_global_bounds()

    def __len__(self):
        return len(self.samples)
        
    def _compute_global_bounds(self, samples_per_class=50):
        pooled_amplitudes = []
        counts = {0: 0, 1: 0, 2: 0, 3: 0}
        
        subset = list(self.samples)
        random.shuffle(subset)
        
        fs = self.target_fs  
        fc = 1.0      
        frequencies = np.linspace(fs/2, 50, 128) 
        scales = (fc * fs) / frequencies
        wavelet = 'cmor5.0-1.0' 
        k = 2
        
        for segment, label in subset:
            if counts[label] >= samples_per_class:
                continue
                
            coefficients, _ = pywt.cwt(segment, scales, wavelet, sampling_period=1/fs)
            amplitude = np.abs(coefficients)
            amplitude = np.power(amplitude, k)
            
            pooled_amplitudes.append(amplitude.flatten())
            counts[label] += 1
            
            if all(counts[c] >= samples_per_class for c in counts):
                break
                
        all_pixels = np.concatenate(pooled_amplitudes)
        global_min = np.percentile(all_pixels, 1)
        global_max = np.percentile(all_pixels, 99)
        
        return (global_min, global_max)

    def generate_cwt(self, signal):
        fs = self.target_fs  
        fc = 1.0      
        frequencies = np.linspace(fs/2, 50, 128) 
        scales = (fc * fs) / frequencies
        wavelet = 'cmor5.0-1.0' 
        
        coefficients, _ = pywt.cwt(signal, scales, wavelet, sampling_period=1/fs)
        amplitude = np.abs(coefficients)
        
        k = 2
        amplitude = np.power(amplitude, k)
        
        amp_min, amp_max = self.amp_bounds
        amplitude_clipped = np.clip(amplitude, amp_min, amp_max)
        
        if amp_max > amp_min:
            normalized_amp = (amplitude_clipped - amp_min) / (amp_max - amp_min)
        else:
            normalized_amp = amplitude_clipped
            
        img_8bit = np.uint8(normalized_amp * 255)
        img_color = cv2.applyColorMap(img_8bit, cv2.COLORMAP_JET)
        img_resized = cv2.resize(img_color, (224, 224))
        return cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        signal_segment, label = self.samples[idx]
        image_np = self.generate_cwt(signal_segment)
        
        if self.transform:
            image_tensor = self.transform(image_np)
        else:
            transform_default = transforms.Compose([transforms.ToTensor()])
            image_tensor = transform_default(image_np)
            
        return image_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
def load_ims_data(set1_path, set2_path, num_files=64):
    data_list, labels = [], []
    
    def get_files(path):
        files = [os.path.join(path, f) for f in os.listdir(path) if not f.startswith('.')]
        return sorted(files)
    
    set1_files = get_files(set1_path)
    set2_files = get_files(set2_path)
    

    for f in set1_files[:num_files]:
        df = pd.read_csv(f, sep='\t', header=None, engine='python')
        if df.shape[1] == 1: df = pd.read_csv(f, delim_whitespace=True, header=None)
        data_list.append(df.iloc[:, 0].values)
        labels.append(0)

    for f in set1_files[-num_files:]:
        df = pd.read_csv(f, sep='\t', header=None, engine='python')
        if df.shape[1] == 1: df = pd.read_csv(f, delim_whitespace=True, header=None)
        data_list.append(df.iloc[:, 4].values)
        labels.append(1)

    for f in set1_files[-num_files:]:
        df = pd.read_csv(f, sep='\t', header=None, engine='python')
        if df.shape[1] == 1: df = pd.read_csv(f, delim_whitespace=True, header=None)
        data_list.append(df.iloc[:, 6].values)
        labels.append(2)

    for f in set2_files[-num_files:]:
        df = pd.read_csv(f, sep='\t', header=None, engine='python')
        if df.shape[1] == 1: df = pd.read_csv(f, delim_whitespace=True, header=None)
        data_list.append(df.iloc[:, 0].values)
        labels.append(3)
        
    return data_list, labels

set1_dir = "/kaggle/input/datasets/onkarraskar/ims-data/IMS_dataset/1st_test/1st_test"
set2_dir = "/kaggle/input/datasets/onkarraskar/ims-data/IMS_dataset/2nd_test/2nd_test"

print("Parsing IMS data files (Extended 10-Hour Window)...")
ims_signals, ims_labels = load_ims_data(set1_dir, set2_dir, num_files=64)


files_by_class = {i: [] for i in range(4)}
for sig, lbl in zip(ims_signals, ims_labels):
    files_by_class[lbl].append(sig)

train_sigs, train_lbls = [], []
val_sigs, val_lbls = [], []
test_sigs, test_lbls = [], []

for lbl, signals in files_by_class.items():
    train_sigs.extend(signals[:22]);    train_lbls.extend([lbl]*22)  
    val_sigs.extend(signals[22:34]);    val_lbls.extend([lbl]*12)    
    test_sigs.extend(signals[34:64]);   test_lbls.extend([lbl]*30)   

ims_train = TargetDataset(train_sigs, train_lbls, original_fs=20000.0, target_fs=6148.44, segment_length=400)
shared_bounds = ims_train.amp_bounds

ims_val   = TargetDataset(val_sigs, val_lbls, original_fs=20000.0, target_fs=6148.44, segment_length=400, amp_bounds=shared_bounds)
ims_test  = TargetDataset(test_sigs, test_lbls, original_fs=20000.0, target_fs=6148.44, segment_length=400, amp_bounds=shared_bounds)

def trim_to_exact_counts(dataset, target_per_class):
    trimmed_samples = []
    counts = {0: 0, 1: 0, 2: 0, 3: 0}
    for segment, label in dataset.samples:
        if counts[label] < target_per_class:
            trimmed_samples.append((segment, label))
            counts[label] += 1
    dataset.samples = trimmed_samples

trim_to_exact_counts(ims_train, target_per_class=81)   
trim_to_exact_counts(ims_val, target_per_class=45)     
trim_to_exact_counts(ims_test, target_per_class=111)   

print("\n" + "="*50)
print("IMS Dataset Split Summary (10-Hour Degradation Test):")
print("="*50)
print(f"Train Segments: {len(ims_train)}")
print(f"Val Segments:   {len(ims_val)}")
print(f"Test Segments:  {len(ims_test)}")
print("="*50 + "\n")

batch_size = 32
ims_train_loader = DataLoader(ims_train, batch_size=batch_size, shuffle=True)
ims_val_loader = DataLoader(ims_val, batch_size=batch_size, shuffle=False)
ims_test_loader = DataLoader(ims_test, batch_size=batch_size, shuffle=False)

In [ ]:
transfer_model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
in_features = transfer_model.classifier[6].in_features

transfer_model.classifier[6] = nn.Linear(in_features, 10)
transfer_model.load_state_dict(torch.load('/kaggle/input/datasets/onkarraskar/cwru-best-weight/cwru_pretrained_vgg16 (1).pth', map_location=device))
print("Successfully loaded pre-trained CWRU baseline weights.")

transfer_model.classifier[6] = nn.Linear(in_features, 4)
transfer_model.classifier[2] = nn.Dropout(p=0.6)
transfer_model.classifier[5] = nn.Dropout(p=0.6)
transfer_model = transfer_model.to(device)

for param in transfer_model.features.parameters():
    param.requires_grad = False
for param in transfer_model.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_stage1 = optim.Adam(transfer_model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs_stage1 = 20
best_val_loss = float('inf')
patience = 5
epochs_no_improve = 0

print("\nStarting Stage 1: Retraining Classifier (IMS - 4 Classes)...")
for epoch in range(num_epochs_stage1):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in ims_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage1.zero_grad()
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage1.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
    
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in ims_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs_stage1:02d}] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'ims_stage1_vgg16.pth')
        print("   --> Saved best Stage 1 model.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"   --> [!] Early stopping triggered at epoch {epoch+1}.")
            break

print("\nStage 1 Complete.")

In [ ]:
transfer_model.load_state_dict(torch.load('/kaggle/working/ims_stage1_vgg16.pth', map_location=device))

for param in transfer_model.features[24:].parameters():
    param.requires_grad = True

optimizer_stage2 = optim.Adam(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-5, weight_decay=1e-4)
best_val_loss = float('inf')
epochs_no_improve = 0
patience = 5

print("\nStarting Stage 2: Fine-Tuning Conv5...")
for epoch in range(15):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in ims_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage2.zero_grad()
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage2.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
    
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in ims_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/15] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'ims_stage2_vgg16.pth')
        print("   --> Saved best Stage 2 model.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("   --> [!] Early stopping triggered.")
            break

In [ ]:
print("==============STAGE 2 COMPLETE. INITIATING TESTING==============")

transfer_model.load_state_dict(torch.load('ims_stage2_vgg16.pth', map_location=device))
transfer_model.eval()

all_preds, all_labels = [], []
test_correct, test_loss, total_test = 0, 0.0, 0

with torch.no_grad():
    for inputs, labels in ims_test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        
        total_test += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / total_test
final_test_acc = 100. * test_correct / total_test

print(f" TEST LOSS: {avg_test_loss:.4f}")
print(f"TEST ACCURACY (IMS Dataset): {final_test_acc:.2f}% \n")

cm = confusion_matrix(all_labels, all_preds)
class_names = ['Healthy', 'Inner Race', 'Roller Element', 'Outer Race']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f' IMS Test Confusion Matrix (Acc: {final_test_acc:.2f}%)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))